# Ruse

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import os
import time
import shutil
import pathlib
import tempfile
from pprint import pprint

import esprima
from esprima_ast_visitor.visitor import EsprimaNodeVisitor

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, HTML, Markdown
import matplot2tikz 
from helpers import *

In [ ]:
class NodeCounter(EsprimaNodeVisitor):
    def __init__(self, avoid_counting=[]):
        super().__init__()
        self.count = 0
        self.avoid_counting = avoid_counting

    # def visit(self, value):
    #     print(type(value))
        # print(f"Node: {node.type}")
    #     # if node.type not in self.avoid_counting:
    #     #     self.count += 1
        # super().visit(value)
        # print()

    def visit_node(self, node):
        if node.type not in self.avoid_counting:
            self.count += 1


def get_ast_node_count(code):
    visitor = NodeCounter(avoid_counting=['Program', 'ExpressionStatement', 'CallExpression'])
    ast = esprima.parse(code, delegate=visitor)
    
    visitor.visit(ast)
    return visitor.count

# print(get_ast_node_count("tree1.value + tree2.value"))
print(get_ast_node_count("tree1.inc_value(), tree2.inc_value(), tree3.inc_value()"))
print(get_ast_node_count("tree1.value + tree2.value + tree3.value"))
print(get_ast_node_count("node_to_delete.right.min_node().swap(node_to_delete).unlink_leaf()"))


## Default experiment configuration

In [ ]:
default_config = {
    "timeout": datetime.timedelta(hours=1),
    "max_iterations": 6,
    "max_sequence_size": 3,
    "max_mutations": 3,
    "max_memory_usage": "100GiB",
    "workers_count": 64
}

## Helper functions

#### Some notebook helper functions

In [ ]:
def display_df(s):
    if isinstance(s, pd.DataFrame):
        s = s.style
    
    s.set_properties(**{'text-align': 'left'})
    s.set_table_styles([dict(selector='th', props=[('text-align', 'left')])])
    
    html_table = s.to_html(justify='left', notebook=True, show_dimensions=True, index=False)

    scrollable_html = f"""
<div style="max-height: 300px; overflow: auto;">
    {html_table}
</div>
"""
    display(HTML(scrollable_html))

class StopExecution(Exception):
    def _render_traceback_(self):
        return []

#### Parse results

In [ ]:
def sort_by_oop_and_side_effects_and_name(df: pd.DataFrame, inplace=False):
    oop_category_order = {
        "Full OOP": 0,
        "Primitive Objects": 1,
        "Primitive": 2,
    }

    side_effects_order = {
        "With Side Effects": 0,
        "Possibly With Side Effects": 1,
        "Without Side Effects": 2
    }

    def sort_func(col):
        if col.name == "oop_category":
            return col.apply(lambda x: oop_category_order[x])
        if col.name == "side_effects":
            return col.apply(lambda x: side_effects_order[x])
        if col.name == "name":
            return col

    return df.sort_values(by=["oop_category", "side_effects", "name"], key=sort_func, inplace=inplace)


def sort_by_found_time(df: pd.DataFrame):
    def sort_func(col):
        if col.name == "found":
            return col.apply(lambda x: 0 if pd.notna(x) else 1)
        if col.name == "total_time":
            return col

    df.sort_values(by=["found", "total_time"], key=sort_func, inplace=True)


def get_sy_info(path):
    with open(path, "r") as f:
        return json.load(f)


def create_tasks_dataframe(tasks, dry_run) -> pd.DataFrame:
    df = pd.DataFrame(tasks)
    df["name"] = df["path"].apply(lambda x: os.path.basename(x).split(".")[0])
    df["total_time"] = df["total_time"].apply(lambda x: pd.Timedelta(
        seconds=x["secs"], nanoseconds=x["nanos"]).total_seconds() if pd.notnull(x) else x).astype(dtype="float64")
    df["path"] = df["path"].apply(lambda x: os.path.relpath(
        os.path.abspath(x), get_workspace_root()))
    df.rename(columns={"total_statistics": "stats"}, inplace=True)

    if "category" in df.columns:
        df["oop_category"] = df["category"].apply(lambda x: x.split(":")[0])
        df["side_effects"] = df["category"].apply(lambda x: x.split(":")[1])
        df.pop("category")

    df = pd.json_normalize(df.to_dict(orient="records"))
    if not dry_run:
        if "found" in df.columns:
            df.pop("found")
        df.rename(columns={"found.found": "found"}, inplace=True)
        sort_by_found_time(df)

    df = df.convert_dtypes(infer_objects=True, convert_string=True,
                           convert_integer=False, convert_boolean=True)

    df['stats.iteration_count'] = df['iterations'].apply(lambda x: len(x))

    ordered_columns = ['name',
                       'total_time',
                       'found',
                       'error',
                       'found.has_side_effects',
                       'found.num_mutations',
                       'found.solution_size',
                       'found.solution_depth',
                       'path',
                       'source',
                       'category',
                       'oop_category',
                       'side_effects',
                       'opcode_count',
                       'max_vm_usage',
                       'stats.Evaluated',
                       'stats.BankSize',
                       'stats.FoundContextCount',
                       'stats.MaxMutatingOpcodes',
                       'stats.MaxDepth',
                       'stats.MaxSize',
                       'stats.iteration_count',
                       'string_literals',
                       'num_literals',
                       'iterations',
                       'start_context.graph_statistics',
                       'start_context.variables',]

    integer_columns = ['found.num_mutations', 'found.solution_size', 'found.solution_depth', 'opcode_count', 'stats.Evaluated',
                       'stats.BankSize', 'stats.FoundContextCount', 'stats.MaxMutatingOpcodes', 'stats.MaxDepth', 'stats.MaxSize',
                       'stats.iteration_count']
    
    for col in integer_columns:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors='coerce', downcast='integer')

    ordered_columns = [col for col in ordered_columns if col in df.columns]
    df = df[ordered_columns]

    return df

def parse_run(results_dir, dry_run):
    tasks = []
    metadata = None
    for file in os.listdir(results_dir):
        if file == 'metadata.json':
            with open(os.path.join(results_dir, file), "r") as f:
                metadata = json.load(f)
        elif file.startswith('task_') and file.endswith('.json'):
            with open(os.path.join(results_dir, file), "r") as f:
                tasks.append(json.load(f))

    df = create_tasks_dataframe(tasks, dry_run)

    df["expected"] = df["path"].apply(lambda x: get_sy_info(
        os.path.join(get_workspace_root(), x))["solution"]["expected"])
    df["examples_count"] = df["path"].apply(lambda x: len(
        get_sy_info(os.path.join(get_workspace_root(), x))["examples"]))
    df["var_count"] = df["path"].apply(lambda x: len(
        get_sy_info(os.path.join(get_workspace_root(), x))["variables"]))

    metadata["tasks_count"] = len(df)
    metadata["total_time"] = datetime.timedelta(seconds=df["total_time"].sum())
    metadata["error_count"] = df["error"].notna().sum()

    df["expected.size"] = df["expected"].apply(lambda x: (get_ast_node_count(code) for code in x))


    return metadata, df

def parse_results(results_dir, dry_run):
    tasks = []
    metadata = []
    if os.path.exists(os.path.join(results_dir, 'metadata.json')):
        run_metadata, run_tasks = parse_run(results_dir, dry_run)
        if run_metadata is not None:
            metadata.append(run_metadata)
        tasks.append(run_tasks)
    else:
        for run_dir in os.listdir(results_dir):
            run_path = os.path.join(results_dir, run_dir)
            if os.path.isdir(run_path) and run_dir.startswith('run_'):
                run_metadata, run_tasks = parse_run(run_path, dry_run)
                if run_metadata is not None:
                    metadata.append(run_metadata)
                tasks.append(run_tasks)

    return metadata, pd.concat(tasks, ignore_index=True)

In [ ]:
class TaskParserError(Exception):
    def __init__(self, errors):
        self.errors = errors
        
    def _render_traceback_(self):
        return []


def get_all_tasks(tasks):
    errors = []

    with tempfile.TemporaryDirectory() as temp_dir:
        results_dir = os.path.join(temp_dir, "results")
        log = os.path.join(temp_dir, "log.jsonl")

        run_ruse(tasks, results_dir, log_file=log, dry_run=True, in_background=False)
        metadata, tasks = parse_results(results_dir, dry_run=True)

        with open(log, "r") as f:
            for line in f:
                parsed = json.loads(line)
                if parsed["level"] == "ERROR":
                    pprint(parsed)
                    errors.append(parsed)
    if len(errors) > 0:
        raise TaskParserError(errors)

    return sort_by_oop_and_side_effects_and_name(tasks)

In [ ]:
def save_tasks_latex_table(df: pd.DataFrame, filename: str, caption: str, label: str):

    columns = ["name", "side_effects", "examples_count", "var_count"]
    headers = ["Task", "Side Effects", "\#Examples", "\#Variables"]
    column_types = ["l", "c", "c", "c"]
    df = df[columns].copy()

    def side_effects_formatter(x):
        if x == "Without Side Effects":
            return "No"
        elif x == "With Side Effects":
            return "Yes"
        elif x == "Possibly With Side Effects":
            return "Possibly"
        raise ValueError(f"Unknown side effect: {x}")

    df["side_effects"] = df["side_effects"].apply(side_effects_formatter)
    df["name"] = df["name"].apply(lambda x: x.replace("_", "\\_"))

    latex = ""
    latex += f"\\begin{{longtable}}{{| l | c c c |}}\n"
    latex += f"\t\\caption{{{caption}. \label{{{label}}} }} \\\\\n\n"
    
    latex += '\t\\hline\n'
    latex += f'\t\\multicolumn{{{len(df.columns)}}}{{| c |}}{{Begin of Table}} \\\\\n'
    latex += '\t\\hline\n'
    latex += "\t" + " & ".join([f"\\textbf{{{h}}}" for h in headers])
    latex += " \\\\\n"
    latex += '\t\\hline\n'
    latex += '\t\\endfirsthead\n\n'

    latex += '\t\\hline\n'
    latex += f'\t\\multicolumn{{{len(df.columns)}}}{{| c |}}{{Continuation of Table \\ref{{{label}}}}} \\\\\n'
    latex += '\t\\hline\n'
    latex += "\t" + " & ".join([f"\\textbf{{{h}}}" for h in headers])
    latex += " \\\\\n"
    latex += '\t\\hline\n'
    latex += '\t\\endhead\n\n'

    latex += '\t\\hline\n'
    latex += '\t\\endfoot\n'
    latex += '\t\\hline\\hline\n'
    latex += '\t\\endlastfoot\n\n'

    for _, row in df.iterrows():
        latex += "\t" + " & ".join([f"{cell}" for cell in row])
        latex += " \\\\\n"
    latex += f"\\end{{longtable}}\n"
    with open(filename, "w") as f:
        f.write(latex)

def save_figure_as_latex(filename: str, **kwargs):
    tikz_plot = matplot2tikz.get_tikz_code(filepath=filename, **kwargs)
    latex = tikz_plot
    with open(filename, "w") as f:
        f.write(latex)

## Experiments

### Benchmarks

We divide the benchmarks into two categories:
* Tasks that use only primitive and built-in objects (Like Array and Set)
* Tasks that use custom User classes

In [ ]:
tasks_paths = [
    os.path.join(get_workspace_root(), "tasks/benchmarks/"),
]

try:
    tasks = get_all_tasks(tasks_paths)
except Exception as e:
    if isinstance(e, TaskParserError):
        raise StopExecution
    else: 
        raise e

display_df(tasks[["name", "side_effects", "expected"]])

tasks_to_ignore = [
    "sypet_10_scale",
    "mut_min_ops_graph_cycle",
    "mut_min_ops_graph_one_way_connected",
    "mut_min_ops_graph",
    "getElementById",
    "set_temp_value"
]

tasks.drop(tasks[tasks["name"].isin(tasks_to_ignore)].index, inplace=True)

full_oop_tasks = tasks[tasks["oop_category"] == "Full OOP"]
primitive_tasks = tasks[tasks["oop_category"] != "Full OOP"]

#### Tasks sources

In [ ]:
ruse_source_tasks = tasks[tasks["path"].str.contains("new_ruse")]
frangel_source_tasks = tasks[tasks["path"].str.contains("fromFrangel")]
sobeq_source_tasks = tasks[tasks["path"].str.contains("fromSobeq")]

print(f"{len(ruse_source_tasks)} Ruse tasks")
print(f"{len(frangel_source_tasks)} FrAngel tasks")
print(f"{len(sobeq_source_tasks)} SoBEq tasks")
print(f"{len(tasks)} tasks")

pure_full_oop_tasks = tasks[(tasks["side_effects"] == "Without Side Effects") & (tasks["oop_category"] == "Full OOP")]
non_pure_full_oop_tasks = tasks[(tasks["side_effects"] != "Without Side Effects") & (tasks["oop_category"] == "Full OOP")]

pure_primitive_objects_tasks = tasks[(tasks["side_effects"] == "Without Side Effects") & (tasks["oop_category"] != "Full OOP")]
non_pure_primitive_objects_tasks = tasks[(tasks["side_effects"] != "Without Side Effects") & (tasks["oop_category"] != "Full OOP")]

print(len(pure_primitive_objects_tasks), len(non_pure_primitive_objects_tasks))
print(len(pure_full_oop_tasks), len(non_pure_full_oop_tasks))



#### Primitive tasks
We compare the primitive tasks against <span style="font-variant:small-caps;">SoBEq</span> and <span style="font-variant:small-caps;">FrAngel</span>

In [ ]:
primitive_table = primitive_tasks.copy()

display_df(primitive_table[["name", "side_effects", "oop_category", "expected"]])
save_tasks_latex_table(primitive_table, "results/figures/primitive_tasks.tex", caption="Primitive and built-in objects tasks", label="primitive_tasks")

pure_primitive_solutions = primitive_table[primitive_table["side_effects"] == "Without Side Effects"]

print(f"{len(primitive_table)} primitive tasks")
print(f"{len(pure_primitive_solutions)} tasks expected solution is pure")
print(f"{len(primitive_table) - len(pure_primitive_solutions)} tasks expected solution can be solved with side effects")

#### Full OOP tasks

We compare the full oop tasks against <span style="font-variant:small-caps;">FrAngel</span>

In [ ]:
full_oop_table = full_oop_tasks.copy()

pure_oop_solutions = full_oop_table[full_oop_table["side_effects"] == "Without Side Effects"]

display_df(full_oop_table[["name", "side_effects", "expected"]])
save_tasks_latex_table(full_oop_table, "results/figures/full_oop_tasks.tex", caption="Full OOP tasks", label="full_oop_tasks")

print(f"{len(full_oop_table)} full oop tasks")
print(f"{len(pure_oop_solutions)} tasks expected solution is pure")
print(f"{len(full_oop_table) - len(pure_oop_solutions)} tasks expected solution can be solved with side effects")


### RQ1 - How does <span style="font-variant:small-caps;">Ruse</span> compare to <span style="font-variant:small-caps;">FrAngel</span> and <span style="font-variant:small-caps;">SoBEq</span>?

In [ ]:
metadata, ruse_tasks = parse_results("results/ruse_benchmarks_results", dry_run=False)
ruse_tasks.to_csv("results/ruse_benchmarks_results.csv", index=False)
ruse_tasks = ruse_tasks.groupby('name').agg(
    total_time = ('total_time', 'mean'),
    total_time_min = ('total_time', 'min'),
    total_time_max = ('total_time', 'max'),
    iteration_count = ('stats.iteration_count', 'first'),
    solved_count=('found', lambda x: x.notnull().sum()),
    found=('found', lambda x: set(x) if x.notnull().any() else None),
    expected=('expected', 'first'),
    solution_size_min=('found.solution_size', 'min'),
    solution_size_max=('found.solution_size', 'max'),
    bank_size=('stats.BankSize', 'mean'),
    bank_size_min=('stats.BankSize', 'min'),
    bank_size_max=('stats.BankSize', 'max'),
    found_context=('stats.FoundContextCount', 'mean'),
    found_context_min=('stats.FoundContextCount', 'min'),
    found_context_max=('stats.FoundContextCount', 'max'),
    iterations=('iterations', 'first'),
    oop_category=('oop_category', 'first'),
    path=('path', 'first'),
    examples_count=('examples_count', 'first'),
    var_count=('var_count', 'first'),
).reset_index()

ruse_compared_tasks = ruse_tasks.copy()
ruse_compared_tasks.drop(ruse_compared_tasks[ruse_compared_tasks["name"].isin(tasks_to_ignore)].index, inplace=True)

ruse_full_oop_tasks = ruse_compared_tasks[ruse_compared_tasks["oop_category"] == "Full OOP"].copy()
ruse_primitive_tasks = ruse_compared_tasks[ruse_compared_tasks["oop_category"] != "Full OOP"].copy()

ruse_full_oop_solved_tasks = ruse_full_oop_tasks[ruse_full_oop_tasks["found"].notnull()]
ruse_primitive_solved_tasks = ruse_primitive_tasks[ruse_primitive_tasks["found"].notnull()]

In [ ]:
display(Markdown(f"#### {len(ruse_compared_tasks)} tasks"))
table = ruse_compared_tasks[["name", "total_time", "found", "expected",
                             "solved_count", "solution_size_min", "solution_size_max"]].copy()
table.sort_values(by="total_time", inplace=True)
# table["expected"] = table["expected"].apply(lambda x: [s.replace('", "', '\n') if pd.notna(s) else s for s in x] if pd.notna(x) else x)


def new_line_format(val, new_line_char="<br>"):
    if isinstance(val, set) or isinstance(val, list):
        formatted = ""
        for v in val:
            if pd.notna(v):
                formatted += f"{v}{new_line_char}"
                
        return formatted.strip()
    return val

table["found_pretty"] = table["found"].apply(new_line_format, new_line_char="<br>")
table["expected_pretty"] = table["expected"].apply(new_line_format, new_line_char="<br>")


def get_good_solutions_mask(data: pd.DataFrame) -> np.ndarray:
    def check_solution_inner(found, expected):
        if pd.isna(found):
            return "No Solution"
        for f in found:
            if pd.notna(f):
                if f.replace('"', "'") in expected:
                    return True
        return False

    return data.apply(lambda x: check_solution_inner(x["found"], x["expected"]), axis=1)

def style_table(s: pd.io.formats.style.Styler):
    def check_solution(data: pd.DataFrame, props: str) -> np.ndarray:
        def check_solution_inner(found, expected):
            if pd.isna(found):
                return np.nan
            for f in found:
                if pd.notna(f):
                    if f.replace('"', "'") in expected:
                        return True
            return False

        good_solutions = get_good_solutions_mask(data).to_numpy()[:, np.newaxis]
        good_solutions = np.repeat(good_solutions, 4, axis=1)
        # print(data)
        return np.where(good_solutions, "", props)

    props = f"background-color: DarkRed;"

    return s.highlight_null(subset="found", color="Brown") \
        .hide(["found", "expected"], axis="columns") \
        .relabel_index(["Task", "Time (s)", "Solved", "Solution min size", "Solution max size", "Found", "Expected"], axis="columns") \
        .apply(check_solution, axis=None, subset=["found", "expected", "found_pretty", "expected_pretty"], props=props)


display_df(table.style.pipe(style_table))

table = ruse_compared_tasks.copy()
table.sort_values(by="total_time", inplace=True)
table["found_pretty"] = table["found"].apply(new_line_format, new_line_char="\n")
table["expected_pretty"] = table["expected"].apply(new_line_format, new_line_char="\n")
table["good_solution"] = get_good_solutions_mask(table)
table.drop(columns=["found", "expected"], inplace=True)
table.rename(columns={"found_pretty": "found", "expected_pretty": "expected"}, inplace=True)
with pd.ExcelWriter("results/ruse_benchmarks.xlsx", engine="xlsxwriter") as writer:
    table.to_excel(writer, index=False, sheet_name='Sheet1')
    workbook  = writer.book
    worksheet = writer.sheets['Sheet1']
    cell_format = workbook.add_format({'text_wrap': True})
    worksheet.set_column('A:Z', cell_format=cell_format)

In [ ]:
display(Markdown(f"#### Solution size"))
display_df(ruse_compared_tasks.sort_values(by="solution_size_min", ascending=False)[["name", "total_time", "found", "solution_size_min"]])

display(Markdown(f"#### Bank size"))
display_df(ruse_compared_tasks.sort_values(by="bank_size_min", ascending=False)[["name", "total_time", "found", "bank_size_min"]])

In [ ]:
def get_real_sy_path(name):
    sobeq_benchmarks_dir = os.path.join(get_workspace_root(), "experiments/ruse_sobeq_benchmarks/benchmarks")
    return pathlib.Path(sobeq_benchmarks_dir).glob(f'**/{os.path.basename(name)}').__next__()

def parse_sobeq_results(path):
    sobeq_tasks = pd.read_csv(path)
    sobeq_tasks = sobeq_tasks[sobeq_tasks["store type"] == "Subsume"]
    sobeq_tasks.pop("store type")
    sobeq_tasks["path"] = sobeq_tasks["name"].apply(get_real_sy_path)
    sobeq_tasks["name"] = sobeq_tasks["path"].apply(lambda x: os.path.basename(x).split(".")[0])
    sobeq_tasks["expected"] = sobeq_tasks["path"].apply(lambda x: get_sy_info(x)["solution"])
    sobeq_tasks["num_examples"] = sobeq_tasks["path"].apply(lambda x: len(get_sy_info(x)["examples"]))
    sobeq_tasks["total_time"] = sobeq_tasks["time (ms)"].apply(lambda x: x / 1000)

    return sobeq_tasks

def parse_frangel_primitive_results(path):
    frangel_tasks = pd.read_csv(path)
    frangel_tasks["path"] = frangel_tasks["name"].apply(get_real_sy_path)
    frangel_tasks["name"] = frangel_tasks["path"].apply(lambda x: os.path.basename(x).split(".")[0])
    frangel_tasks["expected"] = frangel_tasks["path"].apply(lambda x: get_sy_info(x)["solution"])
    frangel_tasks["num_examples"] = frangel_tasks["path"].apply(lambda x: len(get_sy_info(x)["examples"]))
    frangel_tasks["total_time"] = frangel_tasks["time (ms)"].apply(lambda x: x / 1000)

    return frangel_tasks

def parse_frangel_oop_results(path):
    frangel_tasks = pd.read_csv(path)
    # TODO: fix this
    frangel_tasks["num_examples"] = 1
    frangel_tasks["total_time"] = frangel_tasks["time (ms)"].apply(lambda x: x / 1000)
    return frangel_tasks

def get_sobeq_solved_tasks(sobeq_tasks: pd.DataFrame):
    return sobeq_tasks[sobeq_tasks["solution"] != "Timeout"]

def get_frangel_solved_tasks(frangel_tasks: pd.DataFrame):
    frangel_solved_tasks_grouped = frangel_tasks[frangel_tasks["solution"] != "Timeout"].groupby("name")
    def se_minutes(x): return np.std(x/60)/np.sqrt(len(x))
    def se_seconds(x): return np.std(x)/np.sqrt(len(x))
    frangel_solved_tasks = frangel_solved_tasks_grouped.agg(
        num_examples=("num_examples", "first"),
        total_time = ("total_time", "mean"),
        total_time_min = ("total_time", "min"),
        total_time_max = ("total_time", "max"),
        total_time_se_minutes = ("total_time", se_minutes),
        total_time_se_seconds = ("total_time", se_seconds),
    )
    frangel_solved_tasks.reset_index(inplace=True)
    return frangel_solved_tasks


In [ ]:
sobeq_tasks = parse_sobeq_results("results/sobeq_results.csv")
frangel_tasks = parse_frangel_primitive_results("results/frangel_results.csv")
frangel_oop_tasks = parse_frangel_oop_results("results/frangel_full_oop_results.csv")

sobeq_solved_tasks = get_sobeq_solved_tasks(sobeq_tasks)
frangel_solved_tasks = get_frangel_solved_tasks(frangel_tasks)
frangel_oop_solved_tasks = get_frangel_solved_tasks(frangel_oop_tasks)

sobeq_solved_tasks = sobeq_solved_tasks.sort_values(by="total_time")
frangel_solved_tasks = frangel_solved_tasks.sort_values(by="total_time")
frangel_oop_solved_tasks = frangel_oop_solved_tasks.sort_values(by="total_time")
ruse_primitive_solved_tasks = ruse_primitive_solved_tasks.sort_values(by="total_time")
ruse_full_oop_solved_tasks = ruse_full_oop_solved_tasks.sort_values(by="total_time")

In [ ]:
print(f"Ruse solved {len(ruse_primitive_solved_tasks)}/{len(ruse_primitive_tasks)} tasks")
print(f"FrAngel solved {len(frangel_solved_tasks)}/{len(frangel_tasks.groupby('name'))} tasks")
print(f"SObEq solved {len(sobeq_solved_tasks)}/{len(sobeq_tasks)} tasks")
print()
print(f"Ruse OOP solved {len(ruse_full_oop_solved_tasks)}/{len(ruse_full_oop_tasks)} tasks")
print(f"FrAngel OOP solved {len(frangel_oop_solved_tasks)}/{len(frangel_oop_tasks.groupby('name'))} tasks")

In [ ]:
def plot_comparison(solved_tasks: dict[str, pd.DataFrame], colors: dict[str, str], total_tasks: int, y: tuple[list[int], int], x_groups: tuple[int, int, list[int]], figsize=(10, 10), transform_time=None, y_label=None, x_label=None):
    fig, all_axis = plt.subplots(
        1, len(x_groups), figsize=figsize, sharey=True, facecolor='w')
    fig.subplots_adjust(wspace=0.05)
    if len(x_groups) == 1:
        all_axis = [all_axis]

    solved_ticks = []

    yticks, y_max = y

    def get_times(series):
        times = transform_time(series) if transform_time else series
        times = np.array(times)
        times.sort()

        return times

    for ax, (x_start, x_end, xticks) in zip(all_axis, x_groups):
        ax.set_ylim(0, y_max)
        ax.set_xlim(x_start, x_end)
        ax.set_xticks(xticks)

        ax.hlines(y=total_tasks, xmin=x_start, xmax=x_end,
                    color="black", linestyle="--", label="Total tasks")

        for i, (name, df) in enumerate(solved_tasks.items()):
            times = get_times(df["total_time"])
            
            ax.plot(times, np.arange(1, len(times) + 1),
                    color=colors[name], label=name, marker="x")

            if 'total_time_min' in df.columns and 'total_time_max' in df.columns:
                min_times = get_times(df["total_time_min"])
                max_times = get_times(df["total_time_max"])
                
                # Combine min_times and max_times into a single sorted array of unique x values
                combined_times = np.concatenate((min_times, max_times, [x_end]))
                combined_times.sort(kind='mergesort')
                # For each x in combined_times, find the corresponding y value from min_times or max_times
                # If x is in min_times, use its index+1 as y_min; else, use previous y_min
                # Similarly for max_times
                min_times_y = np.searchsorted(min_times, combined_times, side='right')
                max_times_y = np.searchsorted(max_times, combined_times, side='right')

                ax.fill_between(combined_times, max_times_y, min_times_y,
                                color=colors[name], alpha=0.3)
            
            ax.hlines(y=len(times), xmin=times.max(),
                      xmax=x_end, color=colors[name])

            solved_ticks += [len(times)]


    if len(x_groups) == 1:
        ax_l = all_axis[0]
        ax_r = all_axis[0].twinx()
    else:
        ax_l = all_axis[0]
        ax_r = all_axis[-1].twinx()
    
    ax_r.set_ylim(0, y_max)
    ax_r.spines['left'].set_visible(False)
    ax_l.yaxis.tick_left()
    ax_r.yaxis.tick_right()
    ax_l.tick_params(axis='y', labelleft=True)
    ax_r.tick_params(axis='y', labelright=True)
    ax_l.set_yticks(yticks)
    ax_r.set_yticks(np.unique(yticks + solved_ticks))

    if len(x_groups) > 1:
        all_axis[0].spines['right'].set_visible(False)
        for i in range(1, len(all_axis)):
            all_axis[i].spines['right'].set_visible(False)
            all_axis[i].spines['left'].set_visible(False)
            all_axis[i].yaxis.set_visible(False)

        all_axis[-1].spines['left'].set_visible(False)

        d = .015  # how big to make the diagonal lines in axes coordinates
        # arguments to pass plot, just so we don't keep repeating them
        kwargs = dict(
            transform=all_axis[0].transAxes, color='k', clip_on=False)
        all_axis[0].plot((1-d, 1+d), (-d, +d), **kwargs)
        all_axis[0].plot((1-d, 1+d), (1-d, 1+d), **kwargs)
        for i in range(1, len(all_axis)-1):
            kwargs.update(transform=all_axis[i].transAxes)
            all_axis[i].plot((1-d, 1+d), (-d, +d), **kwargs)
            all_axis[i].plot((1-d, 1+d), (1-d, 1+d), **kwargs)
            all_axis[i].plot((-d, +d), (1-d, 1+d), **kwargs)
            all_axis[i].plot((-d, +d), (-d, +d), **kwargs)

        kwargs.update(transform=all_axis[-1].transAxes)
        all_axis[-1].plot((-d, +d), (1-d, 1+d), **kwargs)
        all_axis[-1].plot((-d, +d), (-d, +d), **kwargs)

    all_axis[-1].legend(loc="lower right")

os.makedirs("results/figures", exist_ok=True)

In [ ]:
colors = {
    "frangel": "tab:orange",
    "sobeq": "tab:green",
    "ruse": "tab:blue",
    "frangel_min": ("tab:orange", 0.3),
    "frangel_max": ("tab:orange", 0.3)
}

primitive_solved_tasks = {"frangel": frangel_solved_tasks,
                          "sobeq": sobeq_solved_tasks, 
                          "ruse": ruse_primitive_solved_tasks}
total_primitive_tasks = len(sobeq_tasks)

plot_comparison(primitive_solved_tasks, colors, total_primitive_tasks, 
                transform_time=lambda x: x / 60,
                y=(np.arange(0, 70, 10).tolist() + [total_primitive_tasks], total_primitive_tasks+5),
                x_groups=[(0, 6, np.arange(6)), (6, 65, np.arange(10, 65, 10))],
                figsize=(16, 10))

plt.savefig("results/figures/primitive_solved_tasks.png")
save_figure_as_latex("results/figures/primitive_solved_tasks.tex")

In [ ]:
plot_comparison(primitive_solved_tasks, colors, total_primitive_tasks, 
                transform_time=lambda x: x[x <= 7],
                y=(np.arange(0, 70, 10).tolist() + [total_primitive_tasks], total_primitive_tasks+5),
                x_groups=[(0, 7, np.arange(8))],
                figsize=(16, 10))
plt.savefig("results/figures/primitive_solved_tasks_cutoff.png")

save_figure_as_latex("results/figures/primitive_solved_tasks_cutoff.tex")

In [ ]:
oop_solved_tasks = {"frangel": frangel_oop_solved_tasks,
                    "ruse": ruse_full_oop_solved_tasks}
total_oop_tasks = len(ruse_full_oop_tasks)

plot_comparison(oop_solved_tasks, colors, total_oop_tasks,  
                y=((np.arange(0, 20, 5).tolist() + [total_oop_tasks]), 27),
                x_groups=[(0, 65, np.arange(0, 66, 5))],
                figsize=(16, 10))
plt.ylabel("Number of tasks solved")
plt.xlabel("Time (seconds)")

plt.savefig("results/figures/oop_solved_tasks.png")
save_figure_as_latex("results/figures/oop_solved_tasks.tex")

### RQ2 - Can <span style="font-variant:small-caps;">Ruse</span> handle aliasing and relations between variables?

In [ ]:
relations_tasks = ruse_tasks[ruse_tasks["path"].str.contains("relations")]
display_df(relations_tasks[["name", "expected"]])

In [ ]:
pure_graph = relations_tasks[relations_tasks["name"].str.startswith("graph")].copy()
mut_graph = relations_tasks[relations_tasks["name"].str.startswith("mut_graph")].copy()
mut_min_ops_graph = relations_tasks[relations_tasks["name"].str.startswith("mut_min_ops_graph")].copy()
aliasing = relations_tasks[relations_tasks["name"].isin(["user_names", "user_names_aliasing"])].sort_values(by="name")
connected = relations_tasks[relations_tasks["name"].isin(["user_names", "user_names_connected"])].sort_values(by="name")

def get_bar_name_for_graph_task(task_name):
    suffix = task_name[task_name.find("graph")+6:]
    if suffix == "":
        return "no relation"
    elif suffix == "one_way_connected":
        return "one way connected"
    elif suffix == "cycle":
        return "cycle"
    else:
        raise ValueError(f"Unknown graph task suffix: {suffix}")

def bar_names_order(names: pd.Series):
    return names.apply(lambda name: 0 if name == "no relation" else 1 if name == "one way connected" else 2)

pure_graph["name"] = pure_graph["name"].apply(get_bar_name_for_graph_task)
mut_graph["name"] = mut_graph["name"].apply(get_bar_name_for_graph_task)
mut_min_ops_graph["name"] = mut_min_ops_graph["name"].apply(get_bar_name_for_graph_task)

pure_graph.sort_values(by="name", key=bar_names_order, inplace=True)
mut_graph.sort_values(by="name", key=bar_names_order, inplace=True)
mut_min_ops_graph.sort_values(by="name", key=bar_names_order, inplace=True)

def plot_bar_chart(df, only_one = None):
    if only_one:
        fig = plt.figure(figsize=(16, 8))
    else:
        fig, ax = plt.subplots(1, 3, figsize=(16, 8))

    names = df["name"]
    
    if only_one == "iterations":
        plt.bar(names, df["iterations"].apply(lambda x: len(x)), color="C1")
        plt.title("Iterations")
        plt.locator_params(axis='y', integer=True)
    elif only_one == "bank_size":
        plt.bar(names, df["bank_size"], color="C2")
        plt.title("Bank Size")
        plt.locator_params(axis='y', integer=True)
    elif only_one == "found_context_count":
        plt.bar(names, df["found_context"], color="C3")
        plt.title("Found Context Count")
        plt.locator_params(axis='y', integer=True)
    else:
        ax[0].bar(names, df["iterations"].apply(lambda x: len(x)), color="C1")
        ax[0].title.set_text("Iterations")
        ax[0].locator_params(axis='y', integer=True)

        ax[1].bar(names, df["bank_size"], color="C2")
        ax[1].title.set_text("Bank Size")
        ax[1].locator_params(axis='y', integer=True)

        ax[2].bar(names, df["found_context"], color="C3")
        ax[2].title.set_text("Found Context Count")
        ax[2].locator_params(axis='y', integer=True)

    return fig


In [ ]:
fig = plot_bar_chart(pure_graph)

for subplot in ["iterations", "bank_size", "found_context_count"]:
    fig = plot_bar_chart(pure_graph, subplot)
    save_figure_as_latex(f"results/figures/pure_graph_{subplot}.tex", figure=fig)
    plt.close(fig)

In [ ]:
plot_bar_chart(mut_graph)
for subplot in ["iterations", "bank_size", "found_context_count"]:
    fig = plot_bar_chart(mut_graph, subplot)
    save_figure_as_latex(f"results/figures/mut_graph_{subplot}.tex", figure=fig)
    plt.close(fig)

In [ ]:
plot_bar_chart(mut_min_ops_graph)
for subplot in ["iterations", "bank_size", "found_context_count"]:
    fig = plot_bar_chart(mut_min_ops_graph, subplot)
    save_figure_as_latex(f"results/figures/mut_min_ops_graph_{subplot}.tex", figure=fig)
    plt.close(fig)

In [ ]:
plot_bar_chart(aliasing)
plt.show()

In [ ]:
plot_bar_chart(connected)
plt.show()

### RQ3 - <span style="font-variant:small-caps;">Ruse</span> Embedding overhead

In [ ]:
def read_embedding_result(result_path):
    overhead_runs = []
    for path in os.listdir(result_path):
        if path.endswith(".csv"):
            run = pd.read_csv(os.path.join(result_path, path))
            run["Task"] = run["Task"].apply(lambda x: x.split(".")[0])
            overhead_tasks = tasks.set_index("name").loc[run["Task"]].reset_index()
            run["Time multiplier"] = run["Took with embedding"] / (run["Took without embedding"] * overhead_tasks["examples_count"])
            run["skipped"] = run["Total programs"] - run["Valid sequences"]
            run["skipped%"] = (run["skipped"] / run["Total programs"] * 100)
            overhead_runs.append(run.groupby("Task").last().reset_index())

    def get_program_mil(row):
        return row.mean() / 1_000_000

    overhead = pd.concat(overhead_runs, ignore_index=True).groupby("Task").agg(
        bank_size = ("Bank size", "mean"),
        total_programs = ("Total programs", get_program_mil),
        valid_sequences = ("Valid sequences", get_program_mil),
        took_without_embedding = ("Took without embedding", "mean"),
        took_with_embedding = ("Took with embedding", "mean"),
        time_multiplier = ("Time multiplier", "mean"),
        time_multiplier_std = ("Time multiplier", "std"),
        skipped = ("skipped", get_program_mil),
        skipped_perc = ("skipped%", "mean"),
        skipped_perc_std = ("skipped%", "std"),
    ).reset_index()
    overhead.rename(columns={
        "total_programs": "Total Programs (M)",
        "valid_sequences": "Valid Tuples (M)"
    }, inplace=True)
    overhead["Task"] = overhead["Task"].apply(lambda x: x.split(".")[0])
    overhead_tasks = tasks.set_index("name").loc[overhead["Task"]].reset_index()
    overhead["examples_count"] = overhead_tasks["examples_count"]
    overhead["var_count"] = overhead_tasks["var_count"]
    overhead["nodes"] = overhead_tasks["start_context.graph_statistics"].apply(lambda x: np.mean([g["node_count"] for g in x]).astype("int"))
    overhead["edges"] = overhead_tasks["start_context.graph_statistics"].apply(lambda x: np.mean([g["edge_count"] for g in x]).astype("int"))
    overhead["nodes_per_var"] = overhead_tasks["start_context.variables"].apply(lambda x: list(map(lambda var: np.mean([g["node_count"] for g in var["partial_ctx_graph"]]).astype("int"), x)))
    overhead["oop_category"] = overhead_tasks["oop_category"]

    return overhead

wide_overhead = read_embedding_result("results/ruse_embedding_overhead_results")
display_df(wide_overhead.sort_values(by="time_multiplier", ascending=False)[["Task", "var_count", "examples_count", "nodes", "edges", "nodes_per_var", "skipped_perc", "time_multiplier", "Total Programs (M)", "Valid Tuples (M)", "took_without_embedding", "took_with_embedding"]])

wide_overhead.sort_values(by="time_multiplier", ascending=False)[["Task", "examples_count", "var_count", "nodes", "skipped_perc", "skipped_perc_std", "time_multiplier", "time_multiplier_std"]].to_latex(index=False, float_format="%.2f", buf="results/figures/embedding_overhead.tex")

In [ ]:
full_oop_overhead = wide_overhead[wide_overhead["oop_category"] != "Primitive"]
primitive_overhead = wide_overhead[wide_overhead["oop_category"] == "Primitive"]
print(len(full_oop_overhead), len(primitive_overhead))

# plt.scatter(full_oop_overhead["time_multiplier"], full_oop_overhead["Skipped%"], color="C0")
# plt.scatter(primitive_overhead["time_multiplier"], primitive_overhead["Skipped%"], color="C1")
plt.scatter(wide_overhead["nodes"], wide_overhead["time_multiplier"], color="C0")
plt.scatter(wide_overhead["nodes"] * wide_overhead["examples_count"], wide_overhead["time_multiplier"], color="C1")
plt.scatter(wide_overhead["nodes"] * wide_overhead["examples_count"] * wide_overhead["var_count"], wide_overhead["time_multiplier"], color="C3")
# plt.scatter(wide_overhead["total_programs"], wide_overhead["took_without_embedding"], color="C3")
plt.legend([
    "Nodes",
    "Nodes x Examples",
    "Nodes x Examples x Variables",
])

In [ ]:
graph_overhead = read_embedding_result("results/comparing_graphs_overhead_results")
graph_overhead.sort_values(by="time_multiplier", ascending=False)[["Task", "examples_count", "var_count", "nodes", "skipped_perc", "skipped_perc_std", "time_multiplier", "time_multiplier_std"]].to_latex(index=False, float_format="%.2f", buf="results/figures/graph_embedding_overhead.tex")

display_df(graph_overhead[graph_overhead["Task"].str.len() < 13].sort_values(by="time_multiplier", ascending=False)[["Task", "var_count", "examples_count", "nodes", "edges", "nodes_per_var", "skipped_perc", "time_multiplier", "Total Programs (M)", "Valid Tuples (M)", "took_without_embedding", "took_with_embedding"]])
display_df(graph_overhead[graph_overhead["Task"].str.startswith('graph_one_way_connected')].sort_values(by="time_multiplier", ascending=False)[["Task", "var_count", "examples_count", "nodes", "edges", "nodes_per_var", "skipped_perc", "time_multiplier", "Total Programs (M)", "Valid Tuples (M)", "took_without_embedding", "took_with_embedding"]])
display_df(graph_overhead[graph_overhead["Task"].str.startswith('graph_cycle')].sort_values(by="time_multiplier", ascending=False)[["Task", "var_count", "examples_count", "nodes", "edges", "nodes_per_var", "skipped_perc", "time_multiplier", "Total Programs (M)", "Valid Tuples (M)", "took_without_embedding", "took_with_embedding"]])

examples = ['graph_1', 'graph_2', 'graph_3']
examples_one_way = ['graph_one_way_connected_1', 'graph_one_way_connected_2', 'graph_one_way_connected_3']
examples_cycle = ['graph_cycle_1', 'graph_cycle_2', 'graph_cycle_3']

examples_compare = graph_overhead[graph_overhead["Task"].isin(examples)].sort_values(by="examples_count")
examples_one_way_compare = graph_overhead[graph_overhead["Task"].isin(examples_one_way)].sort_values(by="examples_count")
examples_cycle_compare = graph_overhead[graph_overhead["Task"].isin(examples_cycle)].sort_values(by="examples_count")

plt.scatter(examples_compare["examples_count"], examples_compare["took_with_embedding"], color="C0")
plt.scatter(examples_one_way_compare["examples_count"], examples_one_way_compare["took_with_embedding"], color="C1")
plt.scatter(examples_cycle_compare["examples_count"], examples_cycle_compare["took_with_embedding"], color="C2")

print(examples_compare["took_with_embedding"])
print(examples_one_way_compare["took_with_embedding"])
print(examples_cycle_compare["took_with_embedding"])
coeff = np.polyfit([1, 2, 3], examples_compare["took_with_embedding"], 1)
coeff_one_way = np.polyfit([1, 2, 3], examples_one_way_compare["took_with_embedding"], 1)
coeff_cycle = np.polyfit([1, 2, 3], examples_cycle_compare["took_with_embedding"], 1)
print(coeff, coeff_one_way, coeff_cycle)

plt.legend([
    "graph",
    "graph_one_way_connected",
    "graph_cycle",
])

In [ ]:
wide_overhead = read_embedding_result("results/primitive_overhead_results")
display_df(wide_overhead.sort_values(by="time_multiplier", ascending=False)[["Task", "var_count", "examples_count", "nodes", "edges", "nodes_per_var", "skipped_perc", "time_multiplier", "Total Programs (M)", "Valid Tuples (M)", "took_without_embedding", "took_with_embedding"]])
